# Strong S2UT pilot: Baseline / Monotonic / T2TT

実データのCUDA接続試験 → train/dev alignment診断 → strongの3条件pilot → dev評価を実行します。
元のsplitを保持した同一の部分集合をT2TT/S2UTへ使い、testは使いません。
コーパスは読み取り専用、出力は別のDriveフォルダです。GPUランタイムで上から順に実行してください。
各CLIは失敗時に停止し、ログ・所要時間・学習中のGPU使用率CSVをDriveへ保存します。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, os, subprocess, sys, time
from pathlib import Path

REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'
REPO_REVISION = '10058b5eeba85715b3832a2fa8a5782ab944eda8'
CORPUS = '/content/drive/MyDrive/ja-en-direct-s2st-corpus-data' #@param {type:"string"}
ACCEPTED = CORPUS + '/production/manifests/releases/accepted.jsonl'
ROOT = '/content/drive/MyDrive/alignment-transfer-s2ut-data/pilot-text-001' #@param {type:"string"}
TRAIN_LIMIT = 10000 #@param {type:"integer"}
DEV_LIMIT = 200 #@param {type:"integer"}
SEED = 1 #@param {type:"integer"}
BATCH_SIZE = 8 #@param {type:"integer"}
UPDATE_FREQUENCY = 4 #@param {type:"integer"}
MAX_TOKENS = 20000 #@param {type:"integer"}
LOADER_WORKERS = 2 #@param {type:"integer"}
PREFETCH_BATCHES = 2 #@param {type:"integer"}
LOCAL_CACHE = '/content/alignment-transfer-cache' #@param {type:"string"}
CACHE_GB = 8.0 #@param {type:"number"}
T2TT_UPDATES = 20000 #@param {type:"integer"}
S2UT_UPDATES = 10000 #@param {type:"integer"}
ANNEAL_STEPS = 5000 #@param {type:"integer"}
RUN_CUDA_INTEGRATION = True #@param {type:"boolean"}
GENERATE_WAVEFORMS = False #@param {type:"boolean"}
ASR_BLEU = False #@param {type:"boolean"}

PYTHON = '/content/alignment-venv/bin/python'
REPO = Path('/content/alignment-transfer-s2ut')
DATA = Path(ROOT) / 'data'
MANIFEST = DATA / 'manifest.jsonl'
RUNS = Path(ROOT) / 'runs'
LOGS = Path(ROOT) / 'execution'
CONDITIONS = ['baseline', 'annealed_monotonic', 'annealed_t2tt']
assert not Path(ROOT).resolve().is_relative_to(Path(CORPUS).resolve())
assert Path(ACCEPTED).is_file(), ACCEPTED
assert 1 <= ANNEAL_STEPS < S2UT_UPDATES
assert not ASR_BLEU or GENERATE_WAVEFORMS, 'ASR-BLEU requires generated waveforms'
os.environ['HF_HOME'] = '/content/huggingface'
os.environ['MPLCONFIGDIR'] = '/content/matplotlib-cache'


## 環境の構築と検査

Python 3.10.22 / torch・torchaudio 2.7.1 / 固定fairseqを復元します。
既存ROOTに別のコード・seed・データ条件を混ぜず、変更時は新しいROOTを使ってください。


In [ ]:
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REPO_REVISION], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', REPO_REVISION], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
subprocess.run(['uv', 'venv', '--python', '3.10.22', '--seed', '--allow-existing', '/content/alignment-venv'], check=True)
for packages in [
    ['pip==23.3.2', 'setuptools<81', 'wheel'],
    ['torch==2.7.1', 'torchaudio==2.7.1', 'numpy==1.26.4', 'Cython==3.2.9'],
    ['--no-build-isolation', '-r', 'requirements-fairseq.txt'],
    ['-r', 'requirements.txt'],
]:
    subprocess.run([PYTHON, '-m', 'pip', 'install', *packages], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)

def run(name, module, *arguments, gpu=False, env=None):
    command = [PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'run', '--name', name, '--logs', str(LOGS)]
    if gpu:
        command.append('--monitor-gpu')
    command += ['--', PYTHON, '-m', module, *map(str, arguments)]
    return subprocess.run(command, check=True, env=env)

run('runtime', 'alignment_transfer_s2ut.pilot', 'runtime')
runtime = json.loads(subprocess.check_output([PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'runtime'], text=True))
(Path(ROOT) / 'runtime.json').write_text(json.dumps(runtime, indent=2), encoding='utf-8')
print(runtime)
run('cpu-tests', 'pytest', '-q', '-m', 'not gpu')


## 固定したpilot manifest

train最大10,000件・dev最大200件を元のsplit内でseed固定抽出します。件数は上で変更できます。
音声hashを計算するのは選択したpairだけです。選択結果・元manifestのhash・population件数を保存し、条件変更を伴うresumeを拒否します。


In [ ]:
run('prepare-data', 'alignment_transfer_s2ut.pilot', 'prepare', '--input', ACCEPTED,
    '--corpus-root', CORPUS, '--output', DATA, '--train-limit', TRAIN_LIMIT, '--dev-limit', DEV_LIMIT,
    '--seed', SEED, '--resume')
selection = json.loads((DATA / 'selection.json').read_text())
print({key: selection[key] for key in ['population_counts', 'selected_counts', 'test_used', 'manifest_hash']})
rows = [json.loads(line) for line in MANIFEST.read_text().splitlines() if line.strip()]
run('architecture', 'alignment_transfer_s2ut.compare_architectures', '--t2tt-config', 'configs/t2tt/fisher.yaml',
    '--s2ut-config', 'configs/s2ut/fisher.yaml', '--output', Path(ROOT) / 'architecture.json', '--resume')


## 実モデルのCUDA接続試験

選択済みデータのtrain最大2件・dev1件で実CTC / HuBERT / strongの3条件 / 公式vocoderを確認します。
2更新の接続試験で、翻訳品質の評価ではありません。テスト用出力は試行ごとに別フォルダへ保存します。


In [ ]:
if RUN_CUDA_INTEGRATION:
    tiny = [row for row in rows if row['split'] == 'train'][:2] + [row for row in rows if row['split'] == 'dev'][:1]
    integration = Path(ROOT) / 'integration'
    integration.mkdir(parents=True, exist_ok=True)
    tiny_manifest = integration / 'input.jsonl'
    tiny_manifest.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in tiny), encoding='utf-8')
    environment = os.environ.copy()
    environment['AT_S2UT_GPU_MANIFEST'] = str(tiny_manifest)
    run('cuda-integration', 'pytest', 'tests/test_external_pipeline.py', '-m', 'gpu', '-q', '-s',
        '--basetemp', integration / ('attempt-' + str(time.time_ns())), gpu=True, env=environment)


## T2TTとalignment / unitの前処理

T2TTは同じpilot train pairから別の更新数で学習します。2更新の接続試験用teacherを本pilotへ流用しません。
最良dev lossのT2TT checkpointから、参照訳なしの生成でdev全件のtext BLEUも測定します。BLEU・EOS到達率・原文/予測/参照を確認してから前処理へ進んでください。学習途中のteacherによるpilotから、転移効果の最終結論は出しません。


In [ ]:
run('t2tt', 'alignment_transfer_s2ut.train_t2tt', '--manifest', MANIFEST, '--config', 'configs/t2tt/fisher.yaml',
    '--output', Path(ROOT) / 't2tt', '--max-updates', T2TT_UPDATES, '--warmup-updates', 1000,
    '--seed', SEED, '--batch-size', BATCH_SIZE, '--update-frequency', UPDATE_FREQUENCY, '--max-tokens', MAX_TOKENS,
    '--loader-workers', LOADER_WORKERS, '--prefetch-batches', PREFETCH_BATCHES, '--pin-memory',
    '--validation-interval', 500, '--save-interval-updates', 500, '--device', 'cuda', '--resume-if-exists', gpu=True)
run('t2tt-dev-text', 'alignment_transfer_s2ut.evaluate_t2tt', '--manifest', MANIFEST, '--split', 'dev',
    '--checkpoint', Path(ROOT) / 't2tt/checkpoint_best.pt', '--beam-size', 5, '--max-chars', 1000,
    '--length-penalty', 1.0, '--output', Path(ROOT) / 't2tt/dev-text-metrics.json', '--device', 'cuda', '--resume', gpu=True)
text_metrics = json.loads((Path(ROOT) / 't2tt/dev-text-metrics.json').read_text())
print({key: text_metrics[key] for key in ['text_bleu', 'bleu_signature', 'samples', 'eos_reached_ratio', 'brevity_penalty']})
text_predictions = [json.loads(line) for line in (Path(ROOT) / 't2tt/dev-text-metrics-predictions.jsonl').read_text().splitlines() if line.strip()]
dev_text_by_id = {row['pair_id']: row['en_text'] for row in rows if row['split'] == 'dev'}
for prediction in text_predictions[:5]:
    print('JA:', prediction['source_text'], '\nHYP:', prediction['hypothesis'], '\nREF:', dev_text_by_id[prediction['pair_id']])
run('mt-attention', 'alignment_transfer_s2ut.extract_t2tt_attention', '--manifest', MANIFEST,
    '--checkpoint', Path(ROOT) / 't2tt/checkpoint_best.pt', '--output', Path(ROOT) / 'mt', '--device', 'cuda', '--resume', gpu=True)
run('ctc', 'alignment_transfer_s2ut.align_speech_text', '--manifest', MANIFEST, '--config', 'configs/alignment/ctc.yaml',
    '--output', Path(ROOT) / 'ctc', '--device', 'cuda', '--resume', gpu=True)
run('km100', 'alignment_transfer_s2ut.fetch_assets', '--config', 'configs/units/hubert_km100.yaml')
run('units', 'alignment_transfer_s2ut.extract_units', '--manifest', MANIFEST, '--config', 'configs/units/hubert_km100.yaml',
    '--output', Path(ROOT) / 'units', '--device', 'cuda', '--resume', gpu=True)
run('teacher', 'alignment_transfer_s2ut.build_teacher_alignment', '--manifest', MANIFEST,
    '--mt-attention', Path(ROOT) / 'mt', '--alignment', Path(ROOT) / 'ctc', '--units', Path(ROOT) / 'units',
    '--config', 'configs/s2ut/fisher.yaml', '--output', Path(ROOT) / 'teacher', '--resume')


## Alignment診断：trainとdev

各split最大500件・20 heatmapを保存します。coverage、invalid row、raw / smoothed entropyを確認し、
音声と元の文字列を並べてCTCの対応を確認します。coverageの高さだけで時刻精度が正しいとは判断しません。


In [ ]:
from IPython.display import display, Image, Audio
by_id = {row['pair_id']: row for row in rows}
for split in ['train', 'dev']:
    audit = Path(ROOT) / ('alignment-audit-' + split)
    run('audit-' + split, 'alignment_transfer_s2ut.diagnose_alignment', '--manifest', MANIFEST,
        '--teacher', Path(ROOT) / 'teacher', '--units', Path(ROOT) / 'units', '--config', 'configs/s2ut/fisher.yaml',
        '--split', split, '--max-samples', 500, '--heatmaps', 20, '--seed', SEED, '--source-mapping', 'frontend',
        '--prior-smoothing', 0.05, '--output', audit, '--overwrite')
    summary = json.loads((audit / 'summary.json').read_text())
    print(split, {key: summary[key] for key in ['audited_pairs', 'valid_source_tokens_ratio',
        'valid_target_tokens_ratio', 'valid_unit_rows_ratio', 'prior_entropy', 'smoothed_entropy', 'zero_cell_ratio']})
    assert summary['counts']['valid_unit_rows'] > 0, 'No usable alignment rows; inspect CTC and teacher first'
    pairs = [json.loads(line) for line in (audit / 'pairs.jsonl').read_text().splitlines() if line.strip()]
    # Display two examples per split; all 20 PNG/NPZ pairs remain in the audit directory.
    import hashlib
    for pair in pairs[:2]:
        row = by_id[pair['pair_id']]
        filename = hashlib.sha256(json.dumps(row['pair_id'], ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
        print(row['pair_id'], row['ja_text'], '/', row['en_text'])
        display(Image(filename=str(audit / 'heatmaps' / (filename + '.png'))))
        display(Audio(filename=row['ja_audio']), Audio(filename=row['en_audio']))


## Strongの3条件pilot

全条件で同じ初期化・data・batch・学習率・warmup 1,000・更新数を使います。
既定10,000更新のうち、annealingは5,000更新で終了し、後半はbeta=0です。
planセルは実行予定を保存します。trainセルはcheckpointから自動再開し、失敗した条件で停止します。

少量の入力が揃うと学習を開始し、CPU workerが次の入力を先読みします。未拡張FBANK/CMVNとunit/teacherはローカルSSDの上限付きcacheへ保存し、3条件で共有します。キャッシュは全件コピーせず、必要時に作ります。SpecAugmentは学習threadでGPU転送前に適用します。


In [ ]:
PILOT_ARGS = ['--manifest', str(MANIFEST), '--config', 'configs/s2ut/fisher.yaml', '--units', str(Path(ROOT) / 'units'),
    '--teacher', str(Path(ROOT) / 'teacher'), '--output', str(RUNS), '--logs', str(LOGS),
    '--max-updates', str(S2UT_UPDATES), '--anneal-steps', str(ANNEAL_STEPS), '--seed', str(SEED),
    '--batch-size', str(BATCH_SIZE), '--update-frequency', str(UPDATE_FREQUENCY), '--max-tokens', str(MAX_TOKENS), '--loader-workers', str(LOADER_WORKERS),
    '--prefetch-batches', str(PREFETCH_BATCHES), '--cache-dir', LOCAL_CACHE, '--cache-gb', str(CACHE_GB)]
plan = subprocess.check_output([PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'plan', *PILOT_ARGS], text=True)
(Path(ROOT) / 'pilot-plan.json').write_text(plan, encoding='utf-8')
print(plan)


In [ ]:
subprocess.run([PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'train', *PILOT_ARGS], check=True)
run('comparison', 'alignment_transfer_s2ut.compare_experiments', *[RUNS / name for name in CONDITIONS],
    '--output', Path(ROOT) / 'fair-comparison.json')


## 共通dev評価と集計

dev全件で同じbeam 5 / max units 1,000 / beta=0を使用します。
既定はunit error rate。冒頭でGENERATE_WAVEFORMSとASR_BLEUを有効にすると、共通の固定vocoder・ASRで音声とASR-BLEUも評価します。
pilot後の調整にdevを使い、最終test評価は別途行います。


In [ ]:
speech_only = Path(ROOT) / 'dev-speech-only.jsonl'
run('dev-input', 'alignment_transfer_s2ut.prepare_inference_input', '--manifest', MANIFEST,
    '--split', 'dev', '--output', speech_only, '--resume')
if GENERATE_WAVEFORMS:
    run('vocoder', 'alignment_transfer_s2ut.fetch_assets', '--config', 'configs/s2ut/vocoder.yaml')
for name in CONDITIONS:
    folder = RUNS / name
    vocoder = ['--vocoder-config', 'configs/s2ut/vocoder.yaml'] if GENERATE_WAVEFORMS else ['--units-only']
    run(name + '-dev-infer', 'alignment_transfer_s2ut.infer', '--input', speech_only,
        '--checkpoint', folder / 'checkpoint_best.pt', '--beam-size', 5, '--max-units', 1000,
        '--device', 'cuda', '--output', folder / 'dev-predictions.jsonl', '--resume', *vocoder, gpu=True)
    asr = ['--asr-config', 'configs/experiments/evaluation.yaml', '--device', 'cuda'] if ASR_BLEU else []
    run(name + '-dev-evaluate', 'alignment_transfer_s2ut.evaluate', '--manifest', MANIFEST, '--split', 'dev',
        '--predictions', folder / 'dev-predictions.jsonl', '--units', Path(ROOT) / 'units',
        '--output', folder / 'dev-metrics.json', '--resume', *asr, gpu=ASR_BLEU)
run('pilot-report', 'alignment_transfer_s2ut.pilot', 'report', '--runs', RUNS, '--logs', LOGS,
    '--t2tt-evaluation', Path(ROOT) / 't2tt/dev-text-metrics.json', '--output', Path(ROOT) / 'pilot-summary.json')
summary = json.loads((Path(ROOT) / 'pilot-summary.json').read_text())
display(summary)
display(Image(filename=str(Path(ROOT) / 'pilot-summary.png')))


## 保存される結果

- `data/selection.json`：元のmanifest hash・抽出条件・元splitと使用件数
- `t2tt/dev-text-metrics.json`, `t2tt/dev-text-metrics-predictions.jsonl`：dev text BLEU・SacreBLEU署名・EOS到達率・生成設定・予測文
- `alignment-audit-train/`, `alignment-audit-dev/`：coverage・entropy・sample別指標・20 heatmapずつ
- `runs/{condition}/`：checkpoint・学習metrics・dev予測と評価
- `execution/`：各試行のコマンド・終了状態・所要秒数・完全ログ・5秒間隔のGPU使用率/メモリCSV
- `runtime.json`：Python・torch・CUDA・fairseq revisionとGPU機種/メモリ
- `fair-comparison.json`, `pilot-summary.json`, `pilot-summary.png`：比較条件検査・最良dev loss・beta・更新数・dev評価・記録時間・dev loss曲線

失敗時はexecutionのログを確認し、同じ設定で該当セルから再開できます。
GPU使用率が低い時間帯を学習ログと照合し、FBANK/I/Oの最適化が必要か判断します。
このノートブックは実行準備です。実コーパス上の効果は、実行して得られた結果で判断してください。

`input/data_wait_seconds`は入力を待った時間、`input/optimization_seconds`はCPU augmentation・GPU転送・forward/backward・optimizerの時間です。validationとcheckpoint保存は含みません。LOCAL_CACHEはVM上の一時領域で、runtime再作成後は必要な分から再取得します。
